In [12]:
import os
import csv
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm import tqdm

import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
#-------------读取训练集,训练集地址已经设定好，下面这段不用修改------------------#
#-----Read the training set, the address of the training set has been set, and the following section does not need to be modified-------#
train_path = "/bohr/train-i5ob/v1"

In [22]:
# 读取数据。
def load_train_data(data_dir='./train/'):
    label_path = os.path.join(data_dir, 'train_labels.csv')
    image_dir = os.path.join(data_dir, 'train_images')

    data = []
    with open(label_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sample_id = row['id']
            data.append({
                'id': sample_id,
                'image_path': os.path.join(image_dir, f'{sample_id}.png'),
                'sum': float(row['sum']),
                'product': float(row['product']),
            })

    print(f'Successfully loaded training records: {len(data)}')
    return data


class MNISTBaselineDataset(Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    @staticmethod
    def load_image(image_path):
        image = Image.open(image_path).convert('L')
        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - 0.1307) / 0.3081
        return image

    def __getitem__(self, idx):
        record = self.records[idx]
        image = self.load_image(record['image_path'])
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor([record['sum'], record['product']], dtype=torch.float32)
        return image, target

from sklearn.model_selection import train_test_split
def create_data_loaders(batch_size=64, num_workers=1, data_dir='./train/', val_size=0.1):
    records = load_train_data(data_dir)
    
    # 随机划分训练集和验证集
    train_records, val_records = train_test_split(records, test_size=val_size, random_state=42)

    train_loader = DataLoader(
        MNISTBaselineDataset(train_records),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        MNISTBaselineDataset(val_records),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader

class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 输入通道为1，输出通道为16
            nn.ReLU(),
            nn.MaxPool2d(2),  # 输出尺寸为 16 x 14 x 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 输入通道为16，输出通道为32
            nn.ReLU(),
            nn.MaxPool2d(2),  # 输出尺寸为 32 x 7 x 7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 输入通道为32，输出通道为64
            nn.ReLU(),
            nn.MaxPool2d(2),  # 输出尺寸为 64 x 3 x 3
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 3 * 3, 128),  # 输入特征数为 64 * 3 * 3
            nn.ReLU(),
            nn.Linear(128, 10),  # 输出特征数为 10（对应数字0-9）
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x



# 训练模型
import torch.nn.functional as F

def validate(model, val_loader, device='cpu'):
    model.eval()  # 设置模型为评估模式
    total_correct_sum = 0
    total_correct_product = 0
    total_samples = 0

    with torch.no_grad():  # 不需要计算梯度
        for images, targets in tqdm(val_loader, desc='Validating', leave=False):
            images = images.to(device)
            targets = targets.to(device)
            images_split = torch.split(images, 28, dim=3)
            images_split = torch.stack(images_split, dim=0)

            batch_size = images.size(0)
            preds = torch.zeros(4, batch_size, 10, device=device)

            for i in range(4):
                split_image = images_split[i]
                preds[i] = F.softmax(model(split_image), dim=1)

            # 预测每个数字
            predicted_numbers = []
            for i in range(batch_size):
                predicted_digits = torch.argmax(preds[:, i], dim=1)  # 找到概率最大的数字
                predicted_numbers.append(predicted_digits.cpu().numpy())

            # 计算和与乘积
            for i in range(batch_size):
                predicted_sum = sum(predicted_numbers[i])
                predicted_product = np.prod(predicted_numbers[i])

                # 真实值
                true_sum = int(targets[i][0].item())
                true_product = int(targets[i][1].item())

                # 计算准确率
                if predicted_sum == true_sum:
                    total_correct_sum += 1
                if predicted_product == true_product:
                    total_correct_product += 1

            total_samples += batch_size

    avg_accuracy_sum = total_correct_sum / total_samples
    avg_accuracy_product = total_correct_product / total_samples
    avg_accuracy = (avg_accuracy_sum + avg_accuracy_product) / 2

    print(f'Validation - Average Accuracy - Sum: {avg_accuracy_sum:.4f}, Product: {avg_accuracy_product:.4f}, Overall: {avg_accuracy:.4f}')
def train(model, train_loader, val_loader, epochs, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    from itertools import product
    
    # 定义范围
    max_sum = 36
    max_product = 6561
    
    # 初始化结果字典
    result = { (i, j): [] for i in range(max_sum + 1) for j in range(max_product + 1) }

# 遍历所有可能的数字组合
    for combination in product(range(10), repeat=4):  # 生成四个数字的组合
        total_sum = sum(combination)  # 计算和
        total_product = combination[0] * combination[1] * combination[2] * combination[3]  # 计算乘积
        result[(total_sum, total_product)].append(combination)
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
    
        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            targets = targets.to(device)
            images_split = torch.split(images, 28, dim=3)
            images_split = torch.stack(images_split, dim=0)
    
            optimizer.zero_grad()
            batch_size = images.size(0)  # 获取当前批次的大小
            preds = torch.zeros(4, batch_size, 10, device=device)
            
            for i in range(4):
                split_image = images_split[i]
                preds[i] = F.softmax(model(split_image), dim=1)
    
            # 计算损失
            loss = 0.0
            for i in range(len(targets)):
                target_sum = int(targets[i][0].item())
                target_product = int(targets[i][1].item())
                
                # 获取所有符合条件的组合
                combinations = result.get((target_sum, target_product), [])
                
                if combinations:
                    # 计算每个组合的概率
                    combination_indices = torch.tensor(combinations, device=device)  # 转换为张量
                    prob = preds[0, i, combination_indices[:, 0]] * preds[1, i, combination_indices[:, 1]] * \
                           preds[2, i, combination_indices[:, 2]] * preds[3, i, combination_indices[:, 3]]
                    
                    # 计算总概率
                    total_probability = prob.sum()
                    # 计算损失
                    loss += -torch.log(total_probability + 1e-10)  # 加上小常数以避免log(0)

            loss /= len(targets)  # 平均损失
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}')
        validate(model, val_loader, device)


def set_random_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
batch_size = 64
epochs = 20

set_random_seed(seed)
train_loader, val_loader = create_data_loaders(batch_size=batch_size, data_dir=train_path)
model = MNISTCNN()
train(model, train_loader, val_loader, epochs=epochs, device=device)

In [ ]:
#-------------读取测试集---------------#“DATA_PATH”是测试集加密后的环境变量，按照如下方式可以在提交后，系统评分时访问测试集，但是选手无法直接下载
#----Read the testing set, “DATA_PATH” is an environment variable for the encrypted test set. After submission, you can access the test set for system scoring in the following manner, but the contestant cannot download it directly.-----#
if os.environ.get('DATA_PATH'):
    test_path = os.environ.get("DATA_PATH") + "/"
else:
    test_path = "./test/"
    print("Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")
    #Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象
    #When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.

In [25]:
# 读取测试数据
class MNISTTestDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = sorted(str(path) for path in Path(image_dir).glob('*.png'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        sample_id = Path(image_path).stem
        image = MNISTBaselineDataset.load_image(image_path)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, sample_id


# 这里用训练好的模型直接回归 sum 和 product。
def predict_and_save(model, image_dir, output_file, device='cpu', batch_size=64, num_workers=1):
    dataset = MNISTTestDataset(image_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    rows = []
    model.eval()
    with torch.no_grad():
        for images, sample_ids in tqdm(loader, desc=f'Predicting {os.path.basename(image_dir)}', leave=False):
            images = images.to(device)
            images_split = torch.split(images, 28, dim=3)  # 假设每个图像的宽度为112
            images_split = torch.stack(images_split, dim=0)

            batch_size = images.size(0)
            preds = torch.zeros(4, batch_size, 10, device=device)

            for i in range(4):
                split_image = images_split[i]
                preds[i] = F.softmax(model(split_image), dim=1)

            # 预测每个数字并计算和与乘积
            for i in range(batch_size):
                predicted_digits = torch.argmax(preds[:, i], dim=1)  # 找到概率最大的数字
                predicted_sum = predicted_digits.sum().item()  # 计算和
                predicted_product = predicted_digits.prod().item()  # 计算乘积
                rows.append({
                    'id': sample_ids[i],
                    'sum': int(predicted_sum),
                    'product': int(predicted_product),
                })

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'sum', 'product'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved predictions to {output_file}')


val_dir = os.path.join(test_path, 'val')
test_dir = os.path.join(test_path, 'test')

for required_dir in [val_dir, test_dir]:
    if not os.path.isdir(required_dir):
        raise FileNotFoundError(f'Missing test directory: {required_dir}')

predict_and_save(model, val_dir, 'submission_val.csv', device=str(device), batch_size=batch_size)
predict_and_save(model, test_dir, 'submission_test.csv', device=str(device), batch_size=batch_size)

In [ ]:
import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} 创建成功!')